# Scatterplot generator

This notebook mirrors the cleaned pie-chart generator structure, but implements the scatterplot parameter schema from the attached Excel sheet. It reads the scatter reference sheet, estimates parameter probabilities, samples synthetic scatter data that can express the requested styles, and exports images, tables, and metadata for Altair, Matplotlib, and Plotly.

In [49]:

#from __future__ import annotations

import json
import math
import shutil
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import altair as alt
import kaleido
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import vl_convert


In [50]:

# Project paths
PROJECT = Path("..").resolve()
DATA_REFERENCES = PROJECT / "data" / "references"
OUTPUTS = PROJECT / "outputs"
OUT_ROOT = OUTPUTS / "generated" / "scatter"

CLEAR_OUTPUT = True
LIBRARIES = ["altair", "matplotlib", "plotly"]
SUBDIRS = ["images", "tables", "meta"]

# Update if needed for your local repo layout
REFERENCE_XLSX = DATA_REFERENCES / "scatter charts correct.xlsx"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


## 1. Style schema from the Excel sheet

In [51]:

STYLE_SPEC = {
    "title_present": {"column": 1, "default": True, "codes": {0: False, 1: True}},
    "title_location": {"column": 2, "default": "center", "codes": {0: "none", 1: "center", 2: "left", 3: "right"}},
    "title_color": {"column": 3, "default": "black", "codes": {0: "black", 1: "orange", 2: "red", 3: "darkgray", 4: None, 5: "green"}},
    "title_size": {"column": 4, "default": "medium", "codes": {0: "medium", 1: "small", 2: "large", 3: None}},
    "subtitle_present": {"column": 5, "default": False, "codes": {0: False, 1: True}},
    "legend_present": {"column": 6, "default": False, "codes": {0: False, 1: True}},
    "legend_title_size": {"column": 7, "default": "none", "codes": {0: "none", 1: "small"}},
    "legend_title_color": {"column": 8, "default": None, "codes": {0: None, 1: "black"}},
    "legend_text_color": {"column": 9, "default": "black", "codes": {0: "black", 1: "same_as_title", 2: None, 3: "darkgray"}},
    "legend_outline": {"column": 10, "default": False, "codes": {0: False, 1: True}},
    "legend_fill": {"column": 11, "default": "none", "codes": {0: "none", 1: "gray_block", 2: "plain"}},
    "legend_orientation": {"column": 12, "default": "right", "codes": {0: "none", 1: "left", 2: "right", 3: "top", 4: "bottom", 5: "top_left", 6: "top_right"}},
    "direct_labels": {"column": 13, "default": "none", "codes": {0: "none", 1: "all", 2: "partial"}},
    "label_content": {"column": 14, "default": "none", "codes": {0: "none", 1: "category"}},
    "label_color": {"column": 15, "default": "black", "codes": {0: "black", 1: "same_as_title"}},
    "chart_outline": {"column": 16, "default": "axes_only", "codes": {0: "none", 1: "axes_only", 2: "full"}},
    "gridlines": {"column": 17, "default": "none", "codes": {0: "none", 1: "both", 2: "horizontal", 3: "both_dense"}},
    "gridline_color": {"column": 18, "default": None, "codes": {0: None, 1: "gray", 2: "black"}},
    "image_outline": {"column": 19, "default": False, "codes": {0: False, 1: True}},
    "background": {"column": 20, "default": "white", "codes": {0: "transparent", 1: "white", 2: "light_gray", 3: "dark_gray", 4: "light_color"}},
    "axis_text_orientation": {"column": 21, "default": "parallel", "codes": {0: "parallel", 1: "horizontal"}},
    "axis_text_color": {"column": 22, "default": "black", "codes": {0: "black", 1: "same_as_title", 2: "green"}},
    "x_scale": {"column": 23, "default": "0_20", "codes": {0: "0_20", 1: "0_50"}},
    "x_tick_step": {"column": 24, "default": 5, "codes": {0: 5, 1: 2}},
    "y_scale": {"column": 25, "default": "0_100", "codes": {0: "0_100", 1: "0_50", 2: "0_1000"}},
    "y_tick_step": {"column": 26, "default": 5, "codes": {0: 2, 1: 5, 2: 100}},
    "scatter_orientation": {"column": 27, "default": "ascending", "codes": {0: "descending", 1: "ascending", 2: "blob", 3: "groups"}},
    "n_groups": {"column": 28, "default": 1, "codes": {1: 1, 2: 2, 3: 3, 4: 5, 5: 6}},
    "point_shape_mode": {"column": 29, "default": "dots", "codes": {0: "dots", 1: "squares", 2: "by_group"}},
    "n_shapes": {"column": 30, "default": 1, "codes": {1: 1, 2: 2, 3: 3, 4: 5, 5: 6}},
    "n_colors": {"column": 31, "default": 1, "codes": {1: 1, 2: 2, 3: 3, 4: 5, 5: 6}},
    "palette_type": {"column": 32, "default": "black", "codes": {0: "black", 1: "dark", 2: "bright"}},
    "n_points_bin": {"column": 33, "default": "10_20", "codes": {0: "0_10", 1: "10_20", 2: "20_30", 3: "30_plus"}},
    "regression_line": {"column": 34, "default": 0, "codes": {0: 0, 1: 1, 2: 2}},
    "regression_line_format": {"column": 35, "default": "solid", "codes": {0: "none", 1: "solid", 2: "dashed", 3: "by_group"}},
    "regression_line_color": {"column": 36, "default": "same_as_points", "codes": {0: "none", 1: "different_from_points", 2: "same_as_points", 3: "light_gray", 4: "bright"}},
    "regression_margin": {"column": 37, "default": False, "codes": {0: False, 1: True}},
    "regression_limits": {"column": 38, "default": False, "codes": {0: False, 1: True}},
}

DEFAULT_PARAM_COUNTS = {
    "title_present": {True: 1, False: 1},
    "title_location": {"center": 1, "left": 1, "right": 1, "none": 1},
    "title_color": {"black": 3, "orange": 1, "red": 1, "darkgray": 1, "green": 1},
    "title_size": {"small": 1, "medium": 3, "large": 1},
    "subtitle_present": {False: 3, True: 1},
    "legend_present": {False: 2, True: 2},
    "legend_title_size": {"none": 2, "small": 1},
    "legend_title_color": {None: 2, "black": 1},
    "legend_text_color": {"black": 2, "same_as_title": 1, None: 1, "darkgray": 1},
    "legend_outline": {False: 3, True: 1},
    "legend_fill": {"none": 2, "gray_block": 1, "plain": 1},
    "legend_orientation": {"none": 1, "left": 1, "right": 2, "top": 1, "bottom": 1, "top_left": 1, "top_right": 1},
    "direct_labels": {"none": 2, "all": 1, "partial": 1},
    "label_content": {"none": 2, "category": 1},
    "label_color": {"black": 2, "same_as_title": 1},
    "chart_outline": {"none": 1, "axes_only": 2, "full": 1},
    "gridlines": {"none": 1, "both": 2, "horizontal": 1, "both_dense": 1},
    "gridline_color": {None: 1, "gray": 2, "black": 1},
    "image_outline": {False: 3, True: 1},
    "background": {"transparent": 1, "white": 3, "light_gray": 1, "dark_gray": 1, "light_color": 1},
    "axis_text_orientation": {"parallel": 2, "horizontal": 1},
    "axis_text_color": {"black": 2, "same_as_title": 1, "green": 1},
    "x_scale": {"0_20": 2, "0_50": 1},
    "x_tick_step": {5: 2, 2: 1},
    "y_scale": {"0_50": 1, "0_100": 2, "0_1000": 1},
    "y_tick_step": {2: 1, 5: 2, 100: 1},
    "scatter_orientation": {"descending": 1, "ascending": 1, "blob": 1, "groups": 1},
    "n_groups": {1: 2, 2: 1, 3: 1, 5: 1, 6: 1},
    "point_shape_mode": {"dots": 2, "squares": 1, "by_group": 1},
    "n_shapes": {1: 2, 2: 1, 3: 1, 5: 1, 6: 1},
    "n_colors": {1: 2, 2: 1, 3: 1, 5: 1, 6: 1},
    "palette_type": {"black": 1, "dark": 2, "bright": 1},
    "n_points_bin": {"0_10": 1, "10_20": 2, "20_30": 1, "30_plus": 1},
    "regression_line": {0: 2, 1: 2, 2: 1},
    "regression_line_format": {"none": 1, "solid": 2, "dashed": 1, "by_group": 1},
    "regression_line_color": {"none": 1, "different_from_points": 1, "same_as_points": 2, "light_gray": 1, "bright": 1},
    "regression_margin": {False: 3, True: 1},
    "regression_limits": {False: 3, True: 1},
}


In [52]:

def new_chart_id(prefix: str = "scatter") -> str:
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"


def save_metadata(meta: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)


def ensure_output_dirs(out_root: Path) -> None:
    if CLEAR_OUTPUT and out_root.exists():
        shutil.rmtree(out_root)
    for sub in SUBDIRS:
        for lib in LIBRARIES:
            (out_root / sub / lib).mkdir(parents=True, exist_ok=True)


def safe_int(value) -> int | None:
    if pd.isna(value):
        return None
    try:
        return int(value)
    except Exception:
        return None


def normalize_probabilities(counts: dict) -> dict:
    total = float(sum(counts.values()))
    if total <= 0:
        raise ValueError("Probability weights must sum to a positive value.")
    return {k: float(v) / total for k, v in counts.items()}


def weighted_choice(rng: np.random.Generator, options: dict):
    values = list(options.keys())
    probs = np.asarray(list(options.values()), dtype=float)
    probs = probs / probs.sum()
    return rng.choice(values, p=probs)


def rgba_to_hex(color) -> str:
    return mcolors.to_hex(color, keep_alpha=False)


def relative_luminance(color) -> float:
    rgb = mcolors.to_rgb(color)
    return 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]


def choose_contrast_text_color(fill_color: str) -> str:
    return "white" if relative_luminance(fill_color) < 0.45 else "black"


def extract_reference_rows(excel_path: Path) -> pd.DataFrame:
    raw = pd.read_excel(excel_path, header=None)
    chart_mask = raw[0].astype(str).str.startswith("scatter_", na=False)
    return raw.loc[chart_mask].copy().reset_index(drop=True)


def parse_reference_styles(reference_rows: pd.DataFrame) -> pd.DataFrame:
    parsed = pd.DataFrame(index=reference_rows.index)

    for key, spec in STYLE_SPEC.items():
        col_idx = spec["column"]

        def convert(value):
            if pd.isna(value):
                return np.nan
            intval = safe_int(value)
            if intval is None:
                return np.nan
            return spec["codes"].get(intval, np.nan)

        parsed[key] = reference_rows[col_idx].map(convert)

    parsed["chart_id"] = reference_rows[0].astype(str)
    return parsed


def build_param_stats_from_reference(parsed_styles: pd.DataFrame, alpha: float = 0.25) -> dict:
    param_stats = {}

    for key in STYLE_SPEC:
        series = parsed_styles[key].dropna()
        counts = {opt: float(alpha * wt) for opt, wt in DEFAULT_PARAM_COUNTS[key].items()}

        for value, value_count in series.value_counts(dropna=True).to_dict().items():
            counts[value] = counts.get(value, 0.0) + float(value_count)

        param_stats[key] = normalize_probabilities(counts)

    return param_stats


def load_reference_styles(excel_path: Path = REFERENCE_XLSX) -> tuple[pd.DataFrame, dict]:
    reference_rows = extract_reference_rows(excel_path)
    parsed_styles = parse_reference_styles(reference_rows)
    param_stats = build_param_stats_from_reference(parsed_styles)
    return parsed_styles, param_stats


## 2. Style sampling and synthetic scatter data sampling

In [53]:

def harmonize_style(style: dict) -> dict:
    style = dict(style)
    defaults = {key: spec["default"] for key, spec in STYLE_SPEC.items()}
    for key, default_value in defaults.items():
        style.setdefault(key, default_value)

    if (not style["title_present"]) or style["title_location"] == "none":
        style["title_present"] = False
        style["title_location"] = "none"
        style["subtitle_present"] = False
        style["title_size"] = "medium"
        style["title_color"] = "black"

    # extra safety: if the title is still present but color/size were encoded as "no title"
    if style["title_present"] and style["title_color"] is None:
        style["title_color"] = "black"

    if style["title_present"] and style["title_size"] is None:
        style["title_size"] = "medium"

    if not style["legend_present"] or style["legend_orientation"] == "none":
        style["legend_present"] = False
        style["legend_orientation"] = "none"
        style["legend_title_size"] = "none"
        style["legend_title_color"] = None
        style["legend_text_color"] = None
        style["legend_outline"] = False
        style["legend_fill"] = "none"

    if style["direct_labels"] == "none" or style["label_content"] == "none":
        style["direct_labels"] = "none"
        style["label_content"] = "none"

    if style["gridlines"] == "none":
        style["gridline_color"] = None

    if style["regression_line"] == 0:
        style["regression_line_format"] = "none"
        style["regression_line_color"] = "none"
        style["regression_margin"] = False
        style["regression_limits"] = False

    if style["point_shape_mode"] != "by_group":
        style["n_shapes"] = 1 if style["point_shape_mode"] in {"dots", "squares"} else style["n_shapes"]

    style["n_groups"] = int(style["n_groups"])
    style["n_shapes"] = int(style["n_shapes"])
    style["n_colors"] = int(style["n_colors"])

    return style


def sample_scatter_style(rng: np.random.Generator, param_stats: dict) -> dict:
    style = {key: weighted_choice(rng, param_stats[key]) for key in STYLE_SPEC}
    return harmonize_style(style)


def n_points_from_bin(rng: np.random.Generator, bin_name: str) -> int:
    mapping = {
        "0_10": rng.integers(6, 11),
        "10_20": rng.integers(10, 21),
        "20_30": rng.integers(20, 31),
        "30_plus": rng.integers(30, 61),
    }
    return int(mapping[bin_name])


def scale_bounds(scale_name: str) -> tuple[float, float]:
    return {
        "0_20": (0.0, 20.0),
        "0_50": (0.0, 50.0),
        "0_50_y": (0.0, 50.0),
        "0_100": (0.0, 100.0),
        "0_1000": (0.0, 1000.0),
    }[scale_name]


def build_palette(n: int, style: dict) -> list[str]:
    palette_type = style.get("palette_type", "dark")

    if palette_type == "black":
        palette = ["#111111", "#333333", "#555555", "#777777", "#999999", "#bbbbbb"]
    elif palette_type == "bright":
        palette = ["#e63946", "#f4a261", "#ffbe0b", "#06d6a0", "#118ab2", "#8338ec", "#ff006e"]
    else:
        palette = ["#1d3557", "#457b9d", "#264653", "#3a5a40", "#6c757d", "#7f5539", "#5c677d"]

    return [palette[i % len(palette)] for i in range(max(1, n))]


def build_shape_sequence(style: dict, n_shapes: int) -> list[str]:
    if style["point_shape_mode"] == "dots":
        return ["circle"]
    if style["point_shape_mode"] == "squares":
        return ["square"]
    base = ["circle", "square", "diamond", "triangle-up", "x", "cross"]
    return base[: max(1, n_shapes)]


def make_title(context: dict) -> str:
    title = f"Scatter of {context['y_label']} vs {context['x_label']}"
    if context.get("group_mode"):
        title += f" by {context['group_mode']}"
    return title


def make_subtitle(context: dict) -> str:
    return f"{context['n_points']} points · {context['n_groups']} group(s) · pattern={context['orientation']}"


def choose_label_indices(df: pd.DataFrame, style: dict, rng: np.random.Generator) -> np.ndarray:
    if style["direct_labels"] == "none":
        return np.array([], dtype=int)
    if style["direct_labels"] == "all":
        return df.index.to_numpy()
    n = min(max(3, int(round(len(df) * 0.30))), len(df))
    return np.sort(rng.choice(df.index.to_numpy(), size=n, replace=False))


def compute_regression_df(df: pd.DataFrame, style: dict, x_col: str, y_col: str) -> pd.DataFrame:
    outputs = []

    if style["regression_line"] == 0:
        return pd.DataFrame(columns=[x_col, "yhat", "y_upper", "y_lower", "line_id", "group"])

    if style["regression_line"] == 1:
        groups = [("overall", df.copy())]
    else:
        group_names = list(df["group"].dropna().unique())
        if len(group_names) >= 2:
            groups = [(str(g), df[df["group"] == g].copy()) for g in group_names[:2]]
        else:
            split = df[x_col] <= df[x_col].median()
            groups = [("segment_1", df[split].copy()), ("segment_2", df[~split].copy())]

    for line_idx, (group_name, gdf) in enumerate(groups, start=1):
        if len(gdf) < 2:
            continue
        x = gdf[x_col].to_numpy(dtype=float)
        y = gdf[y_col].to_numpy(dtype=float)
        slope, intercept = np.polyfit(x, y, 1)
        xs = np.linspace(float(x.min()), float(x.max()), 60)
        yhat = intercept + slope * xs
        resid = y - (intercept + slope * x)
        sigma = float(np.std(resid)) if len(resid) > 1 else 0.0
        band = 0.0 if not style["regression_margin"] else max(sigma, 0.02 * max(1.0, float(y.max())))
        out = pd.DataFrame({
            x_col: xs,
            "yhat": yhat,
            "y_upper": yhat + band,
            "y_lower": yhat - band,
            "line_id": f"line_{line_idx}",
            "group": group_name,
        })
        outputs.append(out)

    return pd.concat(outputs, ignore_index=True) if outputs else pd.DataFrame(columns=[x_col, "yhat", "y_upper", "y_lower", "line_id", "group"])


def sample_scatter_data(rng: np.random.Generator, style: dict) -> tuple[pd.DataFrame, dict]:
    n_points = n_points_from_bin(rng, style["n_points_bin"])
    n_groups = max(1, int(style["n_groups"]))

    x_min, x_max = scale_bounds(style["x_scale"])
    y_scale_key = style["y_scale"]
    y_min, y_max = scale_bounds(y_scale_key)

    groups = [f"Group {i+1}" for i in range(n_groups)]
    point_ids = [f"P{i+1}" for i in range(n_points)]

    group_assign = np.array([groups[i % n_groups] for i in range(n_points)])
    rng.shuffle(group_assign)

    x = np.zeros(n_points)
    y = np.zeros(n_points)

    if style["scatter_orientation"] == "groups":
        x_centers = np.linspace(x_min + 0.15 * (x_max - x_min), x_max - 0.15 * (x_max - x_min), n_groups)
        y_centers = np.linspace(y_min + 0.25 * (y_max - y_min), y_max - 0.25 * (y_max - y_min), n_groups)
        rng.shuffle(y_centers)
        for idx, g in enumerate(groups):
            mask = group_assign == g
            x[mask] = rng.normal(x_centers[idx], 0.06 * (x_max - x_min), mask.sum())
            y[mask] = rng.normal(y_centers[idx], 0.08 * (y_max - y_min), mask.sum())
    else:
        x = rng.uniform(x_min, x_max, n_points)
        x_norm = (x - x_min) / max(1e-9, (x_max - x_min))
        noise = rng.normal(0, 0.10, n_points)
        if style["scatter_orientation"] == "ascending":
            y_norm = 0.10 + 0.78 * x_norm + noise
        elif style["scatter_orientation"] == "descending":
            y_norm = 0.88 - 0.75 * x_norm + noise
        else:
            y_norm = 0.50 + rng.normal(0, 0.18, n_points)
        y = y_min + np.clip(y_norm, 0.02, 0.98) * (y_max - y_min)

        if n_groups > 1:
            offsets = np.linspace(-0.10, 0.10, n_groups)
            for idx, g in enumerate(groups):
                mask = group_assign == g
                y[mask] += offsets[idx] * (y_max - y_min)

    x = np.clip(x, x_min, x_max)
    y = np.clip(y, y_min, y_max)

    color_count = max(1, min(style["n_colors"], n_groups if n_groups > 1 else style["n_colors"]))
    shape_count = max(1, min(style["n_shapes"], n_groups if n_groups > 1 else style["n_shapes"]))

    color_categories = [f"Color {i+1}" for i in range(color_count)]
    shape_categories = [f"Shape {i+1}" for i in range(shape_count)]

    color_group_map = {g: color_categories[i % color_count] for i, g in enumerate(groups)}
    shape_group_map = {g: shape_categories[i % shape_count] for i, g in enumerate(groups)}

    df = pd.DataFrame({
        "point_id": point_ids,
        "x": x,
        "y": y,
        "group": group_assign,
        "color_group": [color_group_map[g] for g in group_assign],
        "shape_group": [shape_group_map[g] for g in group_assign],
        "label": point_ids,
    }).sort_values(["group", "x", "y"]).reset_index(drop=True)

    label_idx = choose_label_indices(df, style, rng)
    df["show_label"] = False
    df.loc[label_idx, "show_label"] = True

    reg_df = compute_regression_df(df, style, x_col="x", y_col="y")

    context = {
        "x_label": "X value",
        "y_label": "Y value",
        "n_points": int(n_points),
        "n_groups": int(n_groups),
        "orientation": style["scatter_orientation"],
        "group_mode": "group",
        "x_range": [float(x_min), float(x_max)],
        "y_range": [float(y_min), float(y_max)],
    }
    return df, reg_df, context


## 3. Shared styling helpers

In [54]:

def get_background_color(style: dict) -> str:
    return {
        "transparent": "none",
        "white": "white",
        "light_gray": "#f1f3f5",
        "dark_gray": "#343a40",
        "light_color": "#eef2ff",
    }.get(style.get("background", "white"), "white")


def get_foreground_color(style: dict) -> str:
    bg = get_background_color(style)
    return "white" if bg not in {"white", "#f1f3f5", "#eef2ff", "none"} else "black"


def get_title_fontsize(style: dict) -> int:
    return {"small": 12, "medium": 16, "large": 20}.get(style.get("title_size", "medium"), 16)


def get_title_alignment(style: dict) -> tuple[float, str]:
    location = style.get("title_location", "center")
    if location == "left":
        return 0.01, "left"
    if location == "right":
        return 0.99, "right"
    return 0.50, "center"


def get_altair_title_anchor(style: dict) -> str:
    location = style.get("title_location", "center")
    if location == "left":
        return "start"
    if location == "right":
        return "end"
    return "middle"


def get_plotly_title_anchor(style: dict) -> tuple[float, str]:
    location = style.get("title_location", "center")
    if location == "left":
        return 0.01, "left"
    if location == "right":
        return 0.99, "right"
    return 0.50, "center"


def get_axis_text_color(style: dict) -> str:
    val = style.get("axis_text_color", "black")
    if val == "same_as_title":
        return style.get("title_color", "black")
    return val


def get_label_color(style: dict) -> str:
    val = style.get("label_color", "black")
    if val == "same_as_title":
        return style.get("title_color", "black")
    return val


def get_legend_text_color(style: dict) -> str | None:
    val = style.get("legend_text_color")
    if val == "same_as_title":
        return style.get("title_color", "black")
    return val


def get_mpl_marker(shape_name: str) -> str:
    return {
        "circle": "o",
        "square": "s",
        "diamond": "D",
        "triangle-up": "^",
        "x": "x",
        "cross": "+",
    }.get(shape_name, "o")


def get_plotly_symbol(shape_name: str) -> str:
    return {
        "circle": "circle",
        "square": "square",
        "diamond": "diamond",
        "triangle-up": "triangle-up",
        "x": "x",
        "cross": "cross",
    }.get(shape_name, "circle")


def get_altair_shape(shape_name: str) -> str:
    return {
        "circle": "circle",
        "square": "square",
        "diamond": "diamond",
        "triangle-up": "triangle-up",
        "x": "cross",
        "cross": "diamond-cross",
    }.get(shape_name, "circle")


def legend_position_altair(style: dict) -> tuple[str | None, str | None]:
    mapping = {
        "left": ("left", "middle"),
        "right": ("right", "middle"),
        "top": ("top", "middle"),
        "bottom": ("bottom", "middle"),
        "top_left": ("left", "top"),
        "top_right": ("right", "top"),
    }
    return mapping.get(style.get("legend_orientation", "right"), ("right", "middle"))


def apply_gridline_state_mpl(ax, style: dict):
    grid = style.get("gridlines", "none")

    raw_grid_color = style.get("gridline_color")
    if raw_grid_color == "gray":
        grid_color = "#b0b0b0"
    elif raw_grid_color == "black":
        grid_color = "black"
    else:
        grid_color = "#d0d0d0"   # safe default instead of None

    if grid == "none":
        ax.grid(False)

    elif grid == "horizontal":
        ax.yaxis.grid(True, color=grid_color, linewidth=0.8, alpha=0.7)
        ax.xaxis.grid(False)

    elif grid == "both":
        ax.grid(True, color=grid_color, linewidth=0.8, alpha=0.7)

    else:  # both_dense
        ax.grid(True, color=grid_color, linewidth=0.9, alpha=0.8)
        ax.minorticks_on()
        ax.grid(True, which="minor", color=grid_color, linewidth=0.45, alpha=0.35)

def apply_outline_mpl(ax, fig, style: dict):
    outline = style.get("chart_outline", "axes_only")
    fg = get_foreground_color(style)
    if outline == "none":
        for side in ["top", "right", "bottom", "left"]:
            ax.spines[side].set_visible(False)
    elif outline == "axes_only":
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_color(fg)
        ax.spines["left"].set_color(fg)
    else:
        for side in ["top", "right", "bottom", "left"]:
            ax.spines[side].set_visible(True)
            ax.spines[side].set_color(fg)
            ax.spines[side].set_linewidth(1.0)

    if style.get("image_outline"):
        fig.patch.set_edgecolor(fg)
        fig.patch.set_linewidth(1.2)


## 4. Scatter renderers

In [55]:

def render_scatter_altair(plot_df: pd.DataFrame, reg_df: pd.DataFrame, title: str, subtitle: str, style: dict) -> alt.Chart:
    bg = get_background_color(style)
    fg = get_foreground_color(style)
    axis_color = get_axis_text_color(style)
    title_text = title if style["title_present"] else None
    if title_text and style["subtitle_present"]:
        title_text = [title, subtitle]

    n_color_keys = list(dict.fromkeys(plot_df["color_group"].tolist()))
    n_shape_keys = list(dict.fromkeys(plot_df["shape_group"].tolist()))
    palette = build_palette(len(n_color_keys), style)
    shape_sequence = build_shape_sequence(style, max(1, len(n_shape_keys)))
    shape_values = [get_altair_shape(s) for s in shape_sequence]

    orient_map = {"left": "left", "right": "right", "top": "top", "bottom": "bottom"}
    legend_orient, _ = legend_position_altair(style)
    legend = None
    if style["legend_present"]:
        legend = alt.Legend(
            orient=orient_map.get(legend_orient, "right"),
            title="Group" if style["legend_title_size"] != "none" else None,
            labelColor=get_legend_text_color(style) or fg,
            titleColor=style["legend_title_color"] or fg,
        )

    color_scale = alt.Scale(domain=n_color_keys, range=palette)
    shape_scale = alt.Scale(domain=n_shape_keys, range=shape_values)

    tick_angle = 0 if style["axis_text_orientation"] == "horizontal" else None
    grid = style["gridlines"] in {"both", "horizontal", "both_dense"}

    x_axis_kwargs = {
        "labelColor": axis_color,
        "titleColor": axis_color,
        "grid": style["gridlines"] in {"both", "both_dense"},
        "tickMinStep": style["x_tick_step"],
    }

    if style["axis_text_orientation"] == "horizontal":
        x_axis_kwargs["labelAngle"] = 0

    y_axis_kwargs = {
        "labelColor": axis_color,
        "titleColor": axis_color,
        "grid": grid,
        "tickMinStep": style["y_tick_step"],
    }

    x_axis = alt.Axis(**x_axis_kwargs)
    y_axis = alt.Axis(**y_axis_kwargs)
    
    base = alt.Chart(plot_df).encode(
        x=alt.X("x:Q", title="X value", scale=alt.Scale(domain=list(scale_bounds(style["x_scale"]))), axis=x_axis),
        y=alt.Y("y:Q", title="Y value", scale=alt.Scale(domain=list(scale_bounds(style["y_scale"]))), axis=y_axis),
        color=alt.Color("color_group:N", scale=color_scale, legend=legend if style["n_colors"] > 1 or style["legend_present"] else None),
        shape=alt.Shape("shape_group:N", scale=shape_scale, legend=legend if style["point_shape_mode"] == "by_group" else None),
        tooltip=["point_id", "group", "x", "y"],
    )

    points = base.mark_point(filled=True, size=75)

    layers = [points]

    label_df = plot_df[plot_df["show_label"]].copy()
    if style["direct_labels"] != "none" and style["label_content"] == "category" and len(label_df) > 0:
        labels = alt.Chart(label_df).mark_text(dx=8, dy=-8, color=get_label_color(style), fontSize=11).encode(
            x="x:Q", y="y:Q", text="label:N"
        )
        layers.append(labels)

    if len(reg_df) > 0:
        line_color_map = {g: c for g, c in zip(n_color_keys, palette)}
        line_base = alt.Chart(reg_df).encode(x="x:Q")
        if style["regression_margin"]:
            area = line_base.mark_area(opacity=0.18, color="#999999").encode(y="y_lower:Q", y2="y_upper:Q")
            layers.append(area)
        dash = [1, 0] if style["regression_line_format"] == "solid" else ([6, 4] if style["regression_line_format"] in {"dashed", "by_group"} else [1, 0])
        if style["regression_line_color"] == "light_gray":
            line = line_base.mark_line(color="#bbbbbb", strokeDash=dash, strokeWidth=2).encode(y="yhat:Q", detail="line_id:N")
        elif style["regression_line_color"] == "bright":
            line = line_base.mark_line(color="#ff006e", strokeDash=dash, strokeWidth=2).encode(y="yhat:Q", detail="line_id:N")
        else:
            line = line_base.mark_line(strokeDash=dash, strokeWidth=2).encode(
                y="yhat:Q",
                detail="line_id:N",
                color=alt.Color("group:N", scale=alt.Scale(domain=list(line_color_map.keys()), range=list(line_color_map.values())), legend=None),
            )
        layers.append(line)
        if style["regression_limits"]:
            upper = line_base.mark_line(color="#888888", opacity=0.7, strokeDash=[2, 3]).encode(y="y_upper:Q", detail="line_id:N")
            lower = line_base.mark_line(color="#888888", opacity=0.7, strokeDash=[2, 3]).encode(y="y_lower:Q", detail="line_id:N")
            layers.extend([upper, lower])

    chart = alt.layer(*layers).properties(width=640, height=420)

    config_kwargs = {
        "title": {
            "anchor": get_altair_title_anchor(style),
            "fontSize": get_title_fontsize(style),
            "color": style.get("title_color") or fg,
            "subtitleColor": fg,
        },
        "axis": {
            "domainColor": fg,
            "tickColor": fg,
            "labelColor": axis_color,
            "titleColor": axis_color,
        },
    }

    if bg != "none":
        config_kwargs["background"] = bg

    if style["chart_outline"] == "full":
        config_kwargs["view"] = {"stroke": fg}

    chart = chart.configure(**config_kwargs)

    if title_text:
        chart = chart.properties(title=title_text)

    return chart

In [56]:

def render_scatter_matplotlib(plot_df: pd.DataFrame, reg_df: pd.DataFrame, title: str, subtitle: str, style: dict):
    bg = get_background_color(style)
    fig, ax = plt.subplots(figsize=(8.3, 5.4))
    fig.patch.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)
    ax.set_facecolor((0, 0, 0, 0) if bg == "none" else bg)

    color_keys = list(dict.fromkeys(plot_df["color_group"].tolist()))
    shape_keys = list(dict.fromkeys(plot_df["shape_group"].tolist()))
    palette = build_palette(len(color_keys), style)
    color_map = {k: palette[i] for i, k in enumerate(color_keys)}
    shape_sequence = build_shape_sequence(style, max(1, len(shape_keys)))
    shape_map = {k: get_mpl_marker(shape_sequence[i % len(shape_sequence)]) for i, k in enumerate(shape_keys)}

    for (_, row) in plot_df.iterrows():
        ax.scatter(
            row["x"], row["y"],
            c=color_map[row["color_group"]],
            marker=shape_map[row["shape_group"]],
            s=60,
            alpha=0.9,
            edgecolors="none",
        )

    if style["direct_labels"] != "none" and style["label_content"] == "category":
        label_df = plot_df[plot_df["show_label"]]
        for (_, row) in label_df.iterrows():
            ax.annotate(row["label"], (row["x"], row["y"]), xytext=(5, 5), textcoords="offset points", fontsize=9, color=get_label_color(style))

    if len(reg_df) > 0:
        line_groups = list(dict.fromkeys(reg_df["group"].tolist()))
        bright_colors = ["#ff006e", "#fb5607", "#8338ec", "#3a86ff"]
        for i, group in enumerate(line_groups):
            gdf = reg_df[reg_df["group"] == group]
            dash = "--" if style["regression_line_format"] in {"dashed", "by_group"} and (style["regression_line"] > 0) else "-"
            if style["regression_line_color"] == "light_gray":
                color = "#bbbbbb"
            elif style["regression_line_color"] == "bright":
                color = bright_colors[i % len(bright_colors)]
            elif style["regression_line_color"] == "different_from_points":
                color = "#444444"
            else:
                color = color_map.get(group, palette[i % len(palette)])
            ax.plot(gdf["x"], gdf["yhat"], linestyle=dash, linewidth=2, color=color)
            if style["regression_margin"]:
                ax.fill_between(gdf["x"], gdf["y_lower"], gdf["y_upper"], color=color, alpha=0.15)
            if style["regression_limits"]:
                ax.plot(gdf["x"], gdf["y_upper"], linestyle=":", linewidth=1.2, color=color, alpha=0.7)
                ax.plot(gdf["x"], gdf["y_lower"], linestyle=":", linewidth=1.2, color=color, alpha=0.7)

    axis_color = get_axis_text_color(style)
    ax.set_xlim(*scale_bounds(style["x_scale"]))
    ax.set_ylim(*scale_bounds(style["y_scale"]))
    ax.set_xlabel("X value", color=axis_color)
    ax.set_ylabel("Y value", color=axis_color)
    ax.tick_params(colors=axis_color)

    x_lo, x_hi = scale_bounds(style["x_scale"])
    y_lo, y_hi = scale_bounds(style["y_scale"])
    ax.set_xticks(np.arange(x_lo, x_hi + 1e-9, style["x_tick_step"]))
    ax.set_yticks(np.arange(y_lo, y_hi + 1e-9, style["y_tick_step"]))

    for label in ax.get_xticklabels():
        label.set_rotation(0 if style["axis_text_orientation"] == "horizontal" else 45)
        label.set_ha("right" if style["axis_text_orientation"] == "parallel" else "center")

    apply_gridline_state_mpl(ax, style)
    apply_outline_mpl(ax, fig, style)

    if style["title_present"]:
        xloc, halign = get_title_alignment(style)
        title_text = title if not style["subtitle_present"] else f"{title}\n{subtitle}"
        ax.set_title(
        title_text,
        loc={"left": "left", "center": "center", "right": "right"}[halign],
        color=style.get("title_color") or "black",
        fontsize=get_title_fontsize(style),
        pad=12,
        )

    if style["legend_present"] and (style["n_groups"] > 1 or style["point_shape_mode"] == "by_group"):
        handles = []
        labels = []
        for group in list(dict.fromkeys(plot_df["group"].tolist())):
            sub = plot_df[plot_df["group"] == group].iloc[0]
            handles.append(plt.Line2D([0], [0], marker=shape_map[sub["shape_group"]], color='w', label=group, markerfacecolor=color_map[sub["color_group"]], markersize=8))
            labels.append(group)
        legend_kwargs = {"frameon": style["legend_outline"] or style["legend_fill"] == "gray_block"}
        if style["legend_fill"] == "gray_block":
            legend_kwargs["facecolor"] = "#e9ecef"
        loc_map = {
            "left": "center left",
            "right": "center left",
            "top": "upper center",
            "bottom": "lower center",
            "top_left": "upper left",
            "top_right": "upper right",
        }
        bbox_map = {
            "left": (-0.02, 0.5),
            "right": (1.02, 0.5),
            "top": (0.5, 1.08),
            "bottom": (0.5, -0.20),
            "top_left": (0.0, 1.05),
            "top_right": (1.0, 1.05),
        }
        legend = ax.legend(handles, labels, title=("Group" if style["legend_title_size"] != "none" else None), loc=loc_map[style["legend_orientation"]], bbox_to_anchor=bbox_map[style["legend_orientation"]], ncol=(len(labels) if style["legend_orientation"] in {"top", "bottom"} else 1), **legend_kwargs)
        if style["legend_title_color"]:
            legend.get_title().set_color(style["legend_title_color"])
        txt_color = get_legend_text_color(style)
        if txt_color:
            for txt in legend.get_texts():
                txt.set_color(txt_color)

    fig.tight_layout()
    return fig


In [57]:

def render_scatter_plotly(plot_df: pd.DataFrame, reg_df: pd.DataFrame, title: str, subtitle: str, style: dict):
    color_keys = list(dict.fromkeys(plot_df["color_group"].tolist()))
    palette = build_palette(len(color_keys), style)
    color_map = {k: palette[i] for i, k in enumerate(color_keys)}
    shape_keys = list(dict.fromkeys(plot_df["shape_group"].tolist()))
    shape_sequence = build_shape_sequence(style, max(1, len(shape_keys)))
    symbol_map = {k: get_plotly_symbol(shape_sequence[i % len(shape_sequence)]) for i, k in enumerate(shape_keys)}

    fig = go.Figure()
    groups = list(dict.fromkeys(plot_df["group"].tolist()))
    for group in groups:
        gdf = plot_df[plot_df["group"] == group].copy()
        showlegend = style["legend_present"]
        custom_text = gdf["label"] if style["direct_labels"] != "none" and style["label_content"] == "category" else [""] * len(gdf)
        textpos = "top center"
        if style["direct_labels"] == "partial":
            custom_text = np.where(gdf["show_label"], gdf["label"], "")
        fig.add_trace(go.Scatter(
            x=gdf["x"], y=gdf["y"], mode="markers+text" if style["direct_labels"] != "none" and style["label_content"] == "category" else "markers",
            text=custom_text,
            textposition=textpos,
            textfont=dict(color=get_label_color(style)),
            name=group,
            showlegend=showlegend,
            marker=dict(color=color_map[gdf.iloc[0]["color_group"]], size=10, symbol=symbol_map[gdf.iloc[0]["shape_group"]], line=dict(width=0)),
            hovertemplate="%{text}<br>x=%{x:.2f}<br>y=%{y:.2f}<extra>" + group + "</extra>",
        ))

    if len(reg_df) > 0:
        bright_colors = ["#ff006e", "#fb5607", "#8338ec", "#3a86ff"]
        for i, group in enumerate(list(dict.fromkeys(reg_df["group"].tolist()))):
            gdf = reg_df[reg_df["group"] == group]
            if style["regression_line_color"] == "light_gray":
                line_color = "#bbbbbb"
            elif style["regression_line_color"] == "bright":
                line_color = bright_colors[i % len(bright_colors)]
            elif style["regression_line_color"] == "different_from_points":
                line_color = "#444444"
            else:
                line_color = color_map.get(group, palette[i % len(palette)])
            dash = "dash" if style["regression_line_format"] in {"dashed", "by_group"} else "solid"
            if style["regression_margin"]:
                fig.add_trace(go.Scatter(x=gdf["x"], y=gdf["y_upper"], mode="lines", line=dict(width=0), hoverinfo="skip", showlegend=False))
                fig.add_trace(go.Scatter(x=gdf["x"], y=gdf["y_lower"], mode="lines", line=dict(width=0), fill="tonexty", fillcolor="rgba(120,120,120,0.15)", hoverinfo="skip", showlegend=False))
            fig.add_trace(go.Scatter(x=gdf["x"], y=gdf["yhat"], mode="lines", line=dict(color=line_color, width=2, dash=dash), name=f"Regression {group}", showlegend=False, hoverinfo="skip"))
            if style["regression_limits"]:
                fig.add_trace(go.Scatter(x=gdf["x"], y=gdf["y_upper"], mode="lines", line=dict(color=line_color, width=1, dash="dot"), showlegend=False, hoverinfo="skip"))
                fig.add_trace(go.Scatter(x=gdf["x"], y=gdf["y_lower"], mode="lines", line=dict(color=line_color, width=1, dash="dot"), showlegend=False, hoverinfo="skip"))

    bg = get_background_color(style)
    paper_bg = "rgba(0,0,0,0)" if bg == "none" else bg
    axis_color = get_axis_text_color(style)
    title_text = None
    if style["title_present"]:
        title_text = title if not style["subtitle_present"] else f"{title}<br><sup>{subtitle}</sup>"
    x_pos, x_anchor = get_plotly_title_anchor(style)

    fig.update_layout(
        title=dict(text=title_text, x=x_pos, xanchor=x_anchor, font=dict(color=style.get("title_color", "black"), size=get_title_fontsize(style))) if title_text else None,
        plot_bgcolor=paper_bg,
        paper_bgcolor=paper_bg,
        font=dict(color=get_foreground_color(style)),
        xaxis=dict(title="X value", range=list(scale_bounds(style["x_scale"])), dtick=style["x_tick_step"], tickangle=0 if style["axis_text_orientation"] == "horizontal" else 45, color=axis_color),
        yaxis=dict(title="Y value", range=list(scale_bounds(style["y_scale"])), dtick=style["y_tick_step"], color=axis_color),
        showlegend=style["legend_present"],
        width=760,
        height=500,
    )

    if style["gridlines"] == "none":
        fig.update_xaxes(showgrid=False)
        fig.update_yaxes(showgrid=False)
    elif style["gridlines"] == "horizontal":
        fig.update_xaxes(showgrid=False)
        fig.update_yaxes(showgrid=True, gridcolor="#b0b0b0" if style.get("gridline_color") == "gray" else "black")
    else:
        gridcolor = "#b0b0b0" if style.get("gridline_color") == "gray" else ("black" if style.get("gridline_color") == "black" else "#d0d0d0")
        fig.update_xaxes(showgrid=True, gridcolor=gridcolor)
        fig.update_yaxes(showgrid=True, gridcolor=gridcolor)

    if style["chart_outline"] == "none":
        fig.update_xaxes(showline=False, zeroline=False)
        fig.update_yaxes(showline=False, zeroline=False)
    elif style["chart_outline"] == "axes_only":
        fig.update_xaxes(showline=True, linewidth=1, linecolor=get_foreground_color(style), mirror=False)
        fig.update_yaxes(showline=True, linewidth=1, linecolor=get_foreground_color(style), mirror=False)
    else:
        fig.update_xaxes(showline=True, linewidth=1, linecolor=get_foreground_color(style), mirror=True)
        fig.update_yaxes(showline=True, linewidth=1, linecolor=get_foreground_color(style), mirror=True)

    if style["image_outline"]:
        fig.update_layout(shapes=[dict(type="rect", xref="paper", yref="paper", x0=0, y0=0, x1=1, y1=1, line=dict(color=get_foreground_color(style), width=1))])

    return fig


## 5. Saving, generation, and example usage

In [58]:

def save_altair_svg(chart, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    chart.save(str(out_path), format="svg")


def save_plotly_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(out_path), format="png", scale=2)


def save_matplotlib_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200, bbox_inches="tight", transparent=(fig.get_facecolor()[-1] == 0 if hasattr(fig.get_facecolor(), "__len__") else False))
    plt.close(fig)


def generate_scatter(
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    param_stats: dict,
    max_tries: int = 25,
) -> dict:
    rng = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("scatter")

    for attempt in range(1, max_tries + 1):
        style = sample_scatter_style(rng, param_stats)
        plot_df, reg_df, context = sample_scatter_data(rng=rng, style=style)
        title = make_title(context)
        subtitle = make_subtitle(context)

        table_path = out_root / "tables" / library / f"{chart_id}.csv"
        meta_path = out_root / "meta" / library / f"{chart_id}.json"
        combined_df = plot_df.copy()
        combined_df.to_csv(table_path, index=False)

        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.svg"
            chart = render_scatter_altair(plot_df, reg_df, title, subtitle, style)
            save_altair_svg(chart, image_path)
        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_scatter_matplotlib(plot_df, reg_df, title, subtitle, style)
            save_matplotlib_png(fig, image_path)
        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_scatter_plotly(plot_df, reg_df, title, subtitle, style)
            save_plotly_png(fig, image_path)
        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id": chart_id,
            "chart_type": "scatter",
            "library": library,
            "dataset_source": dataset_source,
            "image_path": str(image_path),
            "table_path": str(table_path),
            "data_context": context,
            "style": style,
            "created_utc": datetime.utcnow().isoformat() + "Z",
            "attempt": attempt,
        }
        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate a valid scatter plot after {max_tries} attempts.")


def generate_batch(
    out_root: Path,
    dataset_source: str,
    generation_plan: dict,
    param_stats: dict,
    start_seed: int = 1000,
) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []
    seed = start_seed

    for library, n in generation_plan.items():
        for _ in range(n):
            metas.append(
                generate_scatter(
                    out_root=out_root,
                    dataset_source=dataset_source,
                    library=library,
                    rng_seed=seed,
                    param_stats=param_stats,
                )
            )
            seed += 1

    return metas


In [59]:

# Optional run cell
# This only runs if the scatter reference Excel file exists at the path above.

if REFERENCE_XLSX.exists():
    ensure_output_dirs(OUT_ROOT)

    reference_styles, PARAM_STATS = load_reference_styles(REFERENCE_XLSX)

    generation_plan = {
        "altair": 10,
        "matplotlib": 10,
        "plotly": 10,
    }

    metas = generate_batch(
        out_root=OUT_ROOT,
        dataset_source="synthetic_scatter_generator",
        generation_plan=generation_plan,
        param_stats=PARAM_STATS,
        start_seed=1000,
    )

    pd.DataFrame(metas)[["chart_id", "library", "image_path"]].head()
else:
    print("Set REFERENCE_XLSX to your local file, then run this cell.")
    print("Missing REFERENCE_XLSX:", not REFERENCE_XLSX.exists())


Set REFERENCE_XLSX to your local file, then run this cell.
Missing REFERENCE_XLSX: True


## 6. Test run of all parameter options individually

In [60]:

from pathlib import Path
import re

TEST_LIBRARIES = ["matplotlib", "altair", "plotly"]
SCATTER_TESTING_ROOT = OUTPUTS / "generated" / "scatter_testing"
UNIQUE_CANONICAL_VALUES_ONLY = True
shutil.rmtree(SCATTER_TESTING_ROOT, ignore_errors=True)

BASE_STYLE = harmonize_style({key: spec["default"] for key, spec in STYLE_SPEC.items()} | {
    "title_present": True,
    "title_location": "center",
    "title_color": "black",
    "title_size": "medium",
    "subtitle_present": True,
    "legend_present": True,
    "legend_title_size": "small",
    "legend_title_color": "black",
    "legend_text_color": "black",
    "legend_outline": False,
    "legend_fill": "plain",
    "legend_orientation": "right",
    "direct_labels": "partial",
    "label_content": "category",
    "label_color": "black",
    "chart_outline": "axes_only",
    "gridlines": "both",
    "gridline_color": "gray",
    "image_outline": False,
    "background": "white",
    "axis_text_orientation": "parallel",
    "axis_text_color": "black",
    "x_scale": "0_20",
    "x_tick_step": 5,
    "y_scale": "0_100",
    "y_tick_step": 5,
    "scatter_orientation": "ascending",
    "n_groups": 3,
    "point_shape_mode": "by_group",
    "n_shapes": 3,
    "n_colors": 3,
    "palette_type": "bright",
    "n_points_bin": "20_30",
    "regression_line": 1,
    "regression_line_format": "solid",
    "regression_line_color": "same_as_points",
    "regression_margin": True,
    "regression_limits": False,
})


def ensure_scatter_testing_dirs(out_root: Path) -> None:
    for sub in SUBDIRS:
        for lib in LIBRARIES:
            (out_root / sub / lib).mkdir(parents=True, exist_ok=True)


def sanitize_slug(text) -> str:
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9._-]+", "_", text)
    return text.strip("_").lower()


def canonical_values_for_param(spec: dict) -> list:
    values = list(spec["codes"].values())
    if UNIQUE_CANONICAL_VALUES_ONLY:
        out = []
        for v in values:
            if v not in out:
                out.append(v)
        values = out
    return values


def render_single_test_case(style_override: dict, library: str, out_root: Path, case_name: str) -> dict:
    style = harmonize_style(dict(BASE_STYLE) | dict(style_override))
    rng = np.random.default_rng(2026)
    plot_df, reg_df, context = sample_scatter_data(rng, style)
    title = case_name.replace("_", " ").title()
    subtitle = make_subtitle(context)
    chart_id = sanitize_slug(case_name)
    table_path = out_root / "tables" / library / f"{chart_id}.csv"
    meta_path = out_root / "meta" / library / f"{chart_id}.json"
    plot_df.to_csv(table_path, index=False)

    if library == "altair":
        image_path = out_root / "images" / library / f"{chart_id}.svg"
        chart = render_scatter_altair(plot_df, reg_df, title, subtitle, style)
        save_altair_svg(chart, image_path)
    elif library == "matplotlib":
        image_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_scatter_matplotlib(plot_df, reg_df, title, subtitle, style)
        save_matplotlib_png(fig, image_path)
    else:
        image_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_scatter_plotly(plot_df, reg_df, title, subtitle, style)
        save_plotly_png(fig, image_path)

    meta = {
        "chart_id": chart_id,
        "library": library,
        "style_override": style_override,
        "style": style,
        "image_path": str(image_path),
        "table_path": str(table_path),
    }
    save_metadata(meta, meta_path)
    return meta


ensure_scatter_testing_dirs(SCATTER_TESTING_ROOT)
all_test_metas = []

for param_name, spec in STYLE_SPEC.items():
    for value in canonical_values_for_param(spec):
        style_override = {param_name: value}
        case_name = f"{param_name}__{sanitize_slug(value)}"
        for library in TEST_LIBRARIES:
            all_test_metas.append(render_single_test_case(style_override, library, SCATTER_TESTING_ROOT, case_name))

pd.DataFrame(all_test_metas)[["chart_id", "library", "image_path"]].head()


,chart_id,library,image_path
0,title_present__false,matplotlib,C:\Users\Michelle\I2R\outputs\generated\scatte...
1,title_present__false,altair,C:\Users\Michelle\I2R\outputs\generated\scatte...
2,title_present__false,plotly,C:\Users\Michelle\I2R\outputs\generated\scatte...
3,title_present__true,matplotlib,C:\Users\Michelle\I2R\outputs\generated\scatte...
4,title_present__true,altair,C:\Users\Michelle\I2R\outputs\generated\scatte...
